In [25]:
%load_ext autoreload
%autoreload 2

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [26]:
import os

from dotenv import load_dotenv
import dspy

load_dotenv()

openai_key = os.getenv(
        "OPENAI_API_KEY"
    )


# Predict: gpt-4o-mini

In [27]:
prediction_lm = dspy.LM("openai/gpt-4o-mini", api_key=openai_key)

dspy.configure(lm=prediction_lm)

## Heroes

In [28]:
from afan.dataset import load_and_preprocess_dataset

hero_test = load_and_preprocess_dataset(
    dataset_name="hero_test",
    data_dir="../data/test/"
)


### Baseline: base_hero_predict

predictor: Predict

signature: BasicHeroSignature

In [29]:
from afan.prompts.signatures import BasicHeroSignature
from afan.utils import predict

base_hero_predict = predict(
    df=hero_test,
    predictor=dspy.Predict(BasicHeroSignature),
    entity="entity"
)

base_hero_predict.shape[0]

76

### CoT BasicHeroSignature

base_hero_cot

In [30]:
base_hero_cot = predict(
    df=hero_test,
    predictor=dspy.ChainOfThought(BasicHeroSignature),
    entity="entity"
)

base_hero_cot.shape[0]

76

### Predict NarrativeArc

narrative_hero_predict

In [31]:
from afan.prompts.signatures import NarrativeArcSignature

narrative_hero_predict = predict(
    df=hero_test,
    predictor=dspy.Predict(NarrativeArcSignature),
    entity="hero"
)

narrative_hero_predict.shape[0]

76

### CoT NarrativeArc

In [32]:
narrative_hero_cot = predict(
    df=hero_test,
    predictor=dspy.ChainOfThought(NarrativeArcSignature),
    entity="hero"
)

narrative_hero_cot.shape[0]

76

# Judge

In [33]:
judge_lm = dspy.LM("gpt-4.1-mini", api_key=openai_key)

dspy.configure(lm=judge_lm)

In [34]:
from afan.prompts.judges import EntitiesMatch
from afan.utils import judge

## Zero-shot judge

### Predict

In [35]:
judge(
    df=base_hero_predict,
    judge_match=dspy.Predict(EntitiesMatch)
)

Accuracy: 30.26%


,ID,text,entities,predicted_entity,judge_match
0,804,Exxon Mobil Lends Its Support to a Carbon Tax ...,"[Carbon tax, The group]",carbon tax proposal,False
1,366,Mongolians sip 'oxygen cocktails' to cope with...,"[NGOs and Non-profit organisations, Mongolian ...",air purifiers,False
2,722,America Is Doubling Down On Climate Progress T...,"[American state governments, cities, state gov...",Sierra Club,False
3,310,The US produced more energy from renewable sou...,"[the US, Renewables]",renewable energy sources,True
4,869,2 protesters arrested after laying down on tra...,[Washington Governor Jay Inslee D],Governor Jay Inslee,True
...,...,...,...,...,...
71,789,EPA wipes its climate change site as protester...,"[Climate activists, senator]",clean energy jobs,False
72,242,Greta Thunberg Mum as Climate Activist Proclai...,[United Nations],Greta Thunberg,False
73,320,The Pope And Big Oil Agree: You Should Pay Mor...,[a tax on carbon dioxide pollution],carbon tax,True
74,713,Preparing For The Impeachment Of Scott Pruitt:...,"[Congress, Impeachment Of Scott Pruitt]",Congress,True


Errors?:

carbon tax should match carbox tax proposal?

In [36]:
judge(
    df=base_hero_cot,
    judge_match=dspy.Predict(EntitiesMatch)
)

Accuracy: 25.00%


,ID,text,entities,predicted_entity,judge_match
0,804,Exxon Mobil Lends Its Support to a Carbon Tax ...,"[Carbon tax, The group]",carbon tax proposal,False
1,366,Mongolians sip 'oxygen cocktails' to cope with...,"[NGOs and Non-profit organisations, Mongolian ...",air purifiers,False
2,722,America Is Doubling Down On Climate Progress T...,"[American state governments, cities, state gov...",U.S. cities and states,True
3,310,The US produced more energy from renewable sou...,"[the US, Renewables]",renewable energy,True
4,869,2 protesters arrested after laying down on tra...,[Washington Governor Jay Inslee D],Governor Jay Inslee,True
...,...,...,...,...,...
71,789,EPA wipes its climate change site as protester...,"[Climate activists, senator]",clean energy industries,False
72,242,Greta Thunberg Mum as Climate Activist Proclai...,[United Nations],Greta Thunberg,False
73,320,The Pope And Big Oil Agree: You Should Pay Mor...,[a tax on carbon dioxide pollution],carbon tax,True
74,713,Preparing For The Impeachment Of Scott Pruitt:...,"[Congress, Impeachment Of Scott Pruitt]",Scott Pruitt,False


#### Narrative arc

In [37]:
judge(
    df= narrative_hero_predict,
    judge_match=dspy.Predict(EntitiesMatch)
)

Accuracy: 26.32%


,ID,text,entities,predicted_entity,judge_match
0,804,Exxon Mobil Lends Its Support to a Carbon Tax ...,"[Carbon tax, The group]","The Climate Leadership Council, advocating for...",False
1,366,Mongolians sip 'oxygen cocktails' to cope with...,"[NGOs and Non-profit organisations, Mongolian ...",Concerned parents and NGOs advocating for clea...,False
2,722,America Is Doubling Down On Climate Progress T...,"[American state governments, cities, state gov...","U.S. cities, states, and businesses leading th...",True
3,310,The US produced more energy from renewable sou...,"[the US, Renewables]","Renewable energy sources (solar, wind, hydroel...",True
4,869,2 protesters arrested after laying down on tra...,[Washington Governor Jay Inslee D],Shut Down Fossil Fuels (the activist group),False
...,...,...,...,...,...
71,789,EPA wipes its climate change site as protester...,"[Climate activists, senator]","The protesters and climate activists, includin...",True
72,242,Greta Thunberg Mum as Climate Activist Proclai...,[United Nations],Greta Thunberg,False
73,320,The Pope And Big Oil Agree: You Should Pay Mor...,[a tax on carbon dioxide pollution],Pope Francis,False
74,713,Preparing For The Impeachment Of Scott Pruitt:...,"[Congress, Impeachment Of Scott Pruitt]",Members of Congress advocating for accountabil...,False


### Chain of Thought

In [16]:
judge(
    df=base_hero_predict,
    judge_match=dspy.ChainOfThought(EntitiesMatch)
)

Accuracy: 34.21%


,ID,text,entities,predicted_entity,judge_match
0,804,Exxon Mobil Lends Its Support to a Carbon Tax ...,"[Carbon tax, The group]",carbon tax proposal,False
1,366,Mongolians sip 'oxygen cocktails' to cope with...,"[NGOs and Non-profit organisations, Mongolian ...",air purifiers,False
2,722,America Is Doubling Down On Climate Progress T...,"[American state governments, cities, state gov...",Sierra Club,False
3,310,The US produced more energy from renewable sou...,"[the US, Renewables]",renewable energy sources,True
4,869,2 protesters arrested after laying down on tra...,[Washington Governor Jay Inslee D],Governor Jay Inslee,True
...,...,...,...,...,...
71,789,EPA wipes its climate change site as protester...,"[Climate activists, senator]",clean energy jobs,False
72,242,Greta Thunberg Mum as Climate Activist Proclai...,[United Nations],Greta Thunberg,False
73,320,The Pope And Big Oil Agree: You Should Pay Mor...,[a tax on carbon dioxide pollution],carbon tax,True
74,713,Preparing For The Impeachment Of Scott Pruitt:...,"[Congress, Impeachment Of Scott Pruitt]",Congress,True


In [17]:
judge_lm.inspect_history(1)





[2025-05-15T11:22:59.513910]

System message:

Your input fields are:
1. `golds` (list[str])
2. `pred` (str)
Your output fields are:
1. `reasoning` (str)
2. `match` (bool)
All interactions will be structured in the following way, with the appropriate values filled in.

[[ ## golds ## ]]
{golds}

[[ ## pred ## ]]
{pred}

[[ ## reasoning ## ]]
{reasoning}

[[ ## match ## ]]
{match}        # note: the value you produce must be True or False

[[ ## completed ## ]]
In adhering to this structure, your objective is: 
        Determine if the predicted entity is semantically equal to at least one
        of the gold entities


User message:

[[ ## golds ## ]]
["Pruitt", "Envrionmental laws"]

[[ ## pred ## ]]
Environmental Protection Agency (EPA)

Respond with the corresponding output fields, starting with the field `[[ ## reasoning ## ]]`, then `[[ ## match ## ]]` (must be formatted as a valid Python bool), and then ending with the marker for `[[ ## completed ## ]]`.


Response:

[[ ## re

In [18]:
judge(
    df=base_hero_cot,
    judge_match=dspy.ChainOfThought(EntitiesMatch)
)

Accuracy: 34.21%


,ID,text,entities,predicted_entity,judge_match
0,804,Exxon Mobil Lends Its Support to a Carbon Tax ...,"[Carbon tax, The group]",carbon tax proposal,False
1,366,Mongolians sip 'oxygen cocktails' to cope with...,"[NGOs and Non-profit organisations, Mongolian ...",air purifiers,False
2,722,America Is Doubling Down On Climate Progress T...,"[American state governments, cities, state gov...",U.S. cities and states,True
3,310,The US produced more energy from renewable sou...,"[the US, Renewables]",renewable energy,True
4,869,2 protesters arrested after laying down on tra...,[Washington Governor Jay Inslee D],Governor Jay Inslee,True
...,...,...,...,...,...
71,789,EPA wipes its climate change site as protester...,"[Climate activists, senator]",clean energy industries,False
72,242,Greta Thunberg Mum as Climate Activist Proclai...,[United Nations],Greta Thunberg,False
73,320,The Pope And Big Oil Agree: You Should Pay Mor...,[a tax on carbon dioxide pollution],carbon tax,True
74,713,Preparing For The Impeachment Of Scott Pruitt:...,"[Congress, Impeachment Of Scott Pruitt]",Scott Pruitt,False


In [19]:
judge(

    df=narrative_hero_predict,
    judge_match=dspy.ChainOfThought(EntitiesMatch)

)

Accuracy: 40.79%


,ID,text,entities,predicted_entity,judge_match
0,804,Exxon Mobil Lends Its Support to a Carbon Tax ...,"[Carbon tax, The group]","The Climate Leadership Council, advocating for...",True
1,366,Mongolians sip 'oxygen cocktails' to cope with...,"[NGOs and Non-profit organisations, Mongolian ...",Concerned parents and NGOs advocating for clea...,False
2,722,America Is Doubling Down On Climate Progress T...,"[American state governments, cities, state gov...","U.S. cities, states, and businesses leading th...",True
3,310,The US produced more energy from renewable sou...,"[the US, Renewables]","Renewable energy sources (solar, wind, hydroel...",True
4,869,2 protesters arrested after laying down on tra...,[Washington Governor Jay Inslee D],Shut Down Fossil Fuels (the activist group),False
...,...,...,...,...,...
71,789,EPA wipes its climate change site as protester...,"[Climate activists, senator]","The protesters and climate activists, includin...",True
72,242,Greta Thunberg Mum as Climate Activist Proclai...,[United Nations],Greta Thunberg,False
73,320,The Pope And Big Oil Agree: You Should Pay Mor...,[a tax on carbon dioxide pollution],Pope Francis,False
74,713,Preparing For The Impeachment Of Scott Pruitt:...,"[Congress, Impeachment Of Scott Pruitt]",Members of Congress advocating for accountabil...,True


In [20]:
judge(

    df=narrative_hero_cot,
    judge_match=dspy.ChainOfThought(EntitiesMatch)

)

Accuracy: 48.68%


,ID,text,entities,predicted_entity,judge_match
0,804,Exxon Mobil Lends Its Support to a Carbon Tax ...,"[Carbon tax, The group]","The Climate Leadership Council, which advocate...",True
1,366,Mongolians sip 'oxygen cocktails' to cope with...,"[NGOs and Non-profit organisations, Mongolian ...","The residents of Ulaanbaatar, particularly par...",False
2,722,America Is Doubling Down On Climate Progress T...,"[American state governments, cities, state gov...","The collective efforts of U.S. cities, states,...",True
3,310,The US produced more energy from renewable sou...,"[the US, Renewables]","Renewable energy sources (solar, wind, hydroel...",True
4,869,2 protesters arrested after laying down on tra...,[Washington Governor Jay Inslee D],"The activists from Shut Down Fossil Fuels, who...",False
...,...,...,...,...,...
71,789,EPA wipes its climate change site as protester...,"[Climate activists, senator]","The protesters and activists, including notabl...",True
72,242,Greta Thunberg Mum as Climate Activist Proclai...,[United Nations],Greta Thunberg and the youth climate activists.,False
73,320,The Pope And Big Oil Agree: You Should Pay Mor...,[a tax on carbon dioxide pollution],Pope Francis,False
74,713,Preparing For The Impeachment Of Scott Pruitt:...,"[Congress, Impeachment Of Scott Pruitt]",The members of Congress who are urged to hold ...,True


Judge predict vs judge Cot on hero narrative arc

In [21]:
judge(

    df=narrative_hero_predict,
    judge_match=dspy.Predict(EntitiesMatch),

)

Accuracy: 26.32%


,ID,text,entities,predicted_entity,judge_match
0,804,Exxon Mobil Lends Its Support to a Carbon Tax ...,"[Carbon tax, The group]","The Climate Leadership Council, advocating for...",False
1,366,Mongolians sip 'oxygen cocktails' to cope with...,"[NGOs and Non-profit organisations, Mongolian ...",Concerned parents and NGOs advocating for clea...,False
2,722,America Is Doubling Down On Climate Progress T...,"[American state governments, cities, state gov...","U.S. cities, states, and businesses leading th...",True
3,310,The US produced more energy from renewable sou...,"[the US, Renewables]","Renewable energy sources (solar, wind, hydroel...",True
4,869,2 protesters arrested after laying down on tra...,[Washington Governor Jay Inslee D],Shut Down Fossil Fuels (the activist group),False
...,...,...,...,...,...
71,789,EPA wipes its climate change site as protester...,"[Climate activists, senator]","The protesters and climate activists, includin...",True
72,242,Greta Thunberg Mum as Climate Activist Proclai...,[United Nations],Greta Thunberg,False
73,320,The Pope And Big Oil Agree: You Should Pay Mor...,[a tax on carbon dioxide pollution],Pope Francis,False
74,713,Preparing For The Impeachment Of Scott Pruitt:...,"[Congress, Impeachment Of Scott Pruitt]",Members of Congress advocating for accountabil...,False


In [22]:
judge(

    df=narrative_hero_predict,
    judge_match= dspy.ChainOfThought(EntitiesMatch),

)

Accuracy: 40.79%


,ID,text,entities,predicted_entity,judge_match
0,804,Exxon Mobil Lends Its Support to a Carbon Tax ...,"[Carbon tax, The group]","The Climate Leadership Council, advocating for...",True
1,366,Mongolians sip 'oxygen cocktails' to cope with...,"[NGOs and Non-profit organisations, Mongolian ...",Concerned parents and NGOs advocating for clea...,False
2,722,America Is Doubling Down On Climate Progress T...,"[American state governments, cities, state gov...","U.S. cities, states, and businesses leading th...",True
3,310,The US produced more energy from renewable sou...,"[the US, Renewables]","Renewable energy sources (solar, wind, hydroel...",True
4,869,2 protesters arrested after laying down on tra...,[Washington Governor Jay Inslee D],Shut Down Fossil Fuels (the activist group),False
...,...,...,...,...,...
71,789,EPA wipes its climate change site as protester...,"[Climate activists, senator]","The protesters and climate activists, includin...",True
72,242,Greta Thunberg Mum as Climate Activist Proclai...,[United Nations],Greta Thunberg,False
73,320,The Pope And Big Oil Agree: You Should Pay Mor...,[a tax on carbon dioxide pollution],Pope Francis,False
74,713,Preparing For The Impeachment Of Scott Pruitt:...,"[Congress, Impeachment Of Scott Pruitt]",Members of Congress advocating for accountabil...,True


In [23]:
from afan.prompts.judges import EntitiesMatchFewShot

judge(

    df=narrative_hero_predict,
    judge_match= dspy.Predict(EntitiesMatchFewShot),

)

Accuracy: 64.47%


,ID,text,entities,predicted_entity,judge_match
0,804,Exxon Mobil Lends Its Support to a Carbon Tax ...,"[Carbon tax, The group]","The Climate Leadership Council, advocating for...",True
1,366,Mongolians sip 'oxygen cocktails' to cope with...,"[NGOs and Non-profit organisations, Mongolian ...",Concerned parents and NGOs advocating for clea...,True
2,722,America Is Doubling Down On Climate Progress T...,"[American state governments, cities, state gov...","U.S. cities, states, and businesses leading th...",True
3,310,The US produced more energy from renewable sou...,"[the US, Renewables]","Renewable energy sources (solar, wind, hydroel...",True
4,869,2 protesters arrested after laying down on tra...,[Washington Governor Jay Inslee D],Shut Down Fossil Fuels (the activist group),False
...,...,...,...,...,...
71,789,EPA wipes its climate change site as protester...,"[Climate activists, senator]","The protesters and climate activists, includin...",True
72,242,Greta Thunberg Mum as Climate Activist Proclai...,[United Nations],Greta Thunberg,False
73,320,The Pope And Big Oil Agree: You Should Pay Mor...,[a tax on carbon dioxide pollution],Pope Francis,False
74,713,Preparing For The Impeachment Of Scott Pruitt:...,"[Congress, Impeachment Of Scott Pruitt]",Members of Congress advocating for accountabil...,True


In [24]:
judge(

    df=narrative_hero_predict,
    judge_match= dspy.ChainOfThought(EntitiesMatchFewShot),

)

Accuracy: 59.21%


,ID,text,entities,predicted_entity,judge_match
0,804,Exxon Mobil Lends Its Support to a Carbon Tax ...,"[Carbon tax, The group]","The Climate Leadership Council, advocating for...",True
1,366,Mongolians sip 'oxygen cocktails' to cope with...,"[NGOs and Non-profit organisations, Mongolian ...",Concerned parents and NGOs advocating for clea...,True
2,722,America Is Doubling Down On Climate Progress T...,"[American state governments, cities, state gov...","U.S. cities, states, and businesses leading th...",True
3,310,The US produced more energy from renewable sou...,"[the US, Renewables]","Renewable energy sources (solar, wind, hydroel...",True
4,869,2 protesters arrested after laying down on tra...,[Washington Governor Jay Inslee D],Shut Down Fossil Fuels (the activist group),False
...,...,...,...,...,...
71,789,EPA wipes its climate change site as protester...,"[Climate activists, senator]","The protesters and climate activists, includin...",True
72,242,Greta Thunberg Mum as Climate Activist Proclai...,[United Nations],Greta Thunberg,False
73,320,The Pope And Big Oil Agree: You Should Pay Mor...,[a tax on carbon dioxide pollution],Pope Francis,False
74,713,Preparing For The Impeachment Of Scott Pruitt:...,"[Congress, Impeachment Of Scott Pruitt]",Members of Congress advocating for accountabil...,True
